# Phase 3: CH-00 & CH-01 — Chuẩn bị Dữ liệu UDIS-D, Đóng băng Protocol và Thực nghiệm Development trên Colab Tesla T4

**Workshop Paper PARMA-DITAM 2027 · crossstage-stitch · Nhánh: `chuong/p3-execution-plan`**

Notebook này dùng để thực thi toàn bộ quy trình chuẩn bị dữ liệu UDIS-D, đóng băng protocol và chạy development sweep trên môi trường **Google Colab GPU Tesla T4** (`sm_75`).

### Quy trình thực hiện:
1. **Mã nguồn & Môi trường (CH-00):** Nạp gói mã nguồn `crossstage_stitch_source.zip`, xác thực GPU Tesla T4 và sinh biên bản `environment_receipt.json`.
2. **Tải tự động UDIS-D (CH-01):** Sử dụng công cụ `prepare_dev_dataset` tích hợp Google Drive File ID chính thức của tác giả (`testing.zip` ~203 MB, `training.zip` ~1.95 GB), tự động tải qua `gdown`, lưu đệm vào Drive cá nhân và sinh source receipts `testing-source.json`, `training-source.json` chuẩn schema `udis-source-v1`.
3. **Strict Ingestion (CH-01):** Giải mã sang `.ppm` uint8 RGB nguyên bản, kiểm soát chặt chẽ lỗi và xác thực đủ đúng **1.105 cặp ảnh test canonical** (danh nghĩa 1.106; bỏ qua tệp mồ côi `000001` do tác giả thiếu `input1/000001.jpg`).
4. **Đóng băng Protocol (CH-01):** Phân chia `train-fit` (250 cặp) và `train-val` (50+ cặp) bằng đồ thị thành phần liên thông (connected components) để triệt tiêu rò rỉ dữ liệu (data leakage); cố định lưới 48 cấu hình feather và quality policy chuẩn.
5. **Build & CTest trên Tesla T4:** Biên dịch CUDA C++ với kiến trúc 75 (`sm_75`) và xác nhận 15/15 bài test CTest pass.
6. **Development Sweep:** Hỗ trợ nạp nhanh bộ số liệu 15.000 dòng telemetry mà Tuấn đã chạy trên Tesla T4 (`p3_dev25_results`), hoặc thực thi sweep trực tiếp bằng `stitch_harness` / `colab_runner`.
7. **Audit Development & Fixed-Best (Đóng mốc CH-01):** Thẩm định ứng viên reference `S1.00_K2000_R1024_Bfeather`, chọn cấu hình Fixed-Best offline và hoàn tất đóng gói bundle `data/p3/`.
8. **Mở rộng các task kế tiếp của Chương:** Hướng dẫn chạy ngay **Ablation A4 (Same-H Blending)** và **CH-05 (Synthetic Scaling 720p/1080p/4K)** trên Colab T4.

## 1. Nạp Mã Nguồn và Kiểm tra GPU Tesla T4 (CH-00)

1. Chọn **Runtime → Change runtime type → T4 GPU**.
2. **Nạp mã nguồn vào Colab:** Vì repository GitHub là **Private**, cell dưới đây hỗ trợ tự động nhận file nén mã nguồn **`crossstage_stitch_source.zip`** (được đóng gói sẵn bằng `python tools/repro/package_source.py` từ máy tính):
   - Nếu bạn đã kéo thả file vào panel Files (`/content/`) hoặc mount từ Google Drive (`MyDrive/crossstage_stitch_source.zip`), notebook sẽ tự nhận diện và giải nén.
   - Nếu chưa có, notebook sẽ hiển thị nút **Choose Files** để bạn chọn file `crossstage_stitch_source.zip` tải lên trực tiếp.
   - Gói nén này đã tích hợp sẵn cả bộ kết quả `p3_dev25_results` của Tuấn để bạn có thể kiểm thử ngay lập tức mà không cần chạy lại toàn bộ GPU sweep.

In [ ]:
# @title 1.1 Kiểm tra GPU và Nạp Mã Nguồn Dự Án { display-mode: "form" }
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

# 1. Kiểm tra GPU Tesla T4
gpu_check = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True
)
if gpu_check.returncode != 0:
    print("❌ CẢNH BÁO: Chưa phát hiện GPU! Hãy chọn Runtime -> Change runtime type -> T4 GPU.")
else:
    print(f"✅ GPU phát hiện: {gpu_check.stdout.strip()}")

# 2. Mount Google Drive nếu có
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Đã kết nối Google Drive.")
except Exception:
    print("ℹ️ Bỏ qua mount Google Drive (hoạt động hoàn toàn trên ổ đĩa cục bộ Colab).")

# 3. Giải nén hoặc Tải lên crossstage_stitch_source.zip
REPO_DIR = Path("/content/crossstage-stitch")
REPO_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_ZIP = Path("/content/crossstage_stitch_source.zip")
DRIVE_ZIP = Path("/content/drive/MyDrive/crossstage_stitch_source.zip")

if not SOURCE_ZIP.exists() and DRIVE_ZIP.exists():
    print("Copying crossstage_stitch_source.zip từ Google Drive...")
    shutil.copy2(DRIVE_ZIP, SOURCE_ZIP)

if not SOURCE_ZIP.exists():
    print("Chưa thấy crossstage_stitch_source.zip trong /content/.")
    print("Vui lòng chọn file crossstage_stitch_source.zip từ máy tính để tải lên:")
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        if name.endswith(".zip"):
            SOURCE_ZIP = Path("/content") / name
            break

if SOURCE_ZIP.exists():
    print(f"Đang giải nén {SOURCE_ZIP.name} vào {REPO_DIR}...")
    with zipfile.ZipFile(SOURCE_ZIP, "r") as zf:
        zf.extractall(REPO_DIR)
    print(f"✅ Đã giải nén thành công {len(list(REPO_DIR.rglob('*')))} tệp vào {REPO_DIR}")
else:
    print("❌ Không tìm thấy gói mã nguồn!")

os.chdir(REPO_DIR)
print(f"✅ Thư mục làm việc hiện tại: {os.getcwd()}")

# 4. Khởi tạo Git repo cục bộ trên Colab để phục vụ capture environment receipt (CH-00)
if not (REPO_DIR / ".git").exists():
    print("Khởi tạo Git repo cục bộ trên Colab để phục vụ chụp environment receipt...")
    subprocess.run(["git", "init", "-b", "chuong/p3-execution-plan"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "config", "user.name", "chuong"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "config", "user.email", "chuong@crossstage.local"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "add", "."], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "commit", "-m", "chore: colab environment snapshot"], cwd=REPO_DIR, check=True)
    print("✅ Đã khởi tạo Git repo Colab thành công.")


In [ ]:
# @title 1.2 Khởi tạo và Xác minh Colab Environment Receipt (CH-00) { display-mode: "form" }
from pathlib import Path
Path("build/p3-colab").mkdir(parents=True, exist_ok=True)
!python -m tools.env.generate_receipt --output build/p3-colab/environment_receipt.json --verbose
!python -m tools.env.generate_receipt --output build/p3-colab/environment_receipt.json --verify-only


## 2. Chuẩn bị File Nguồn UDIS-D: Tải Tự Động & Lập Source Receipts (CH-01)

### 📌 Nguồn dữ liệu UDIS-D chính thức từ tác giả:
Bộ dữ liệu gồm 2 gói nén được cấu hình sẵn Google Drive File ID trong `tools.data_prep.prepare_dev_dataset`:
- **`testing.zip`** (ID: `1unnBdPg9Vi12-1xXoUdzMub_XUq7yZz0`, ~203 MB): Chứa **1.105 cặp ảnh test canonical** (danh nghĩa 1.106; `input1` thiếu `000001.jpg`, bỏ qua tệp mồ côi `000001`).
- **`training.zip`** (ID: `1CTKa42BRdXO2vdYSadaQkvgQ8DR6l49a`, ~1.95 GB): Chứa 10.440 cặp ảnh training để trích chọn split `train-fit` và `train-val`.

### 💡 Cơ chế tải và lưu đệm thông minh:
- Cell dưới đây kiểm tra Google Drive `MyDrive/UDIS-D/` trước. Nếu bạn đã tải ở phiên trước, notebook chỉ mất 1-2 giây để copy sang ổ đĩa Colab.
- Nếu chưa có, script tự động tải trực tiếp từ Google Drive tác giả bằng `gdown` với băng thông cực cao của Colab (~100-200 MB/s), sau đó lưu một bản đệm vào Drive của bạn.

In [ ]:
# @title 2.1 Tải tự động UDIS-D hoặc Copy từ Google Drive Cache { display-mode: "form" }
from pathlib import Path
import shutil

RAW_DIR = Path("/content/crossstage-stitch/data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)
P3_DIR = Path("/content/crossstage-stitch/data/p3")
P3_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_CACHE = Path("/content/drive/MyDrive/UDIS-D")
if DRIVE_CACHE.parent.exists():
    DRIVE_CACHE.mkdir(parents=True, exist_ok=True)

TRAIN_ZIP = RAW_DIR / "training.zip"
TEST_ZIP = RAW_DIR / "testing.zip"

# 1. Copy từ Google Drive Cache nếu có sẵn
for split, fname in [("testing", "testing.zip"), ("training", "training.zip")]:
    dest = RAW_DIR / fname
    cached = DRIVE_CACHE / fname if DRIVE_CACHE.exists() else None
    if cached and cached.exists() and cached.stat().st_size > 1024 * 1024 and not dest.exists():
        print(f"Đang copy {fname} ({cached.stat().st_size // (1024*1024)} MB) từ Google Drive cache...")
        shutil.copy2(cached, dest)

# 2. Tải tự động qua prepare_dev_dataset nếu data/raw còn thiếu
need_download = []
if not TEST_ZIP.exists():
    need_download.append("testing")
if not TRAIN_ZIP.exists():
    need_download.append("training")

if need_download:
    dl_split = "both" if len(need_download) == 2 else need_download[0]
    print(f"Đang tải tự động {dl_split} qua gdown từ Google Drive tác giả...")
    !python -m tools.data_prep.prepare_dev_dataset \
        --mode gdrive \
        --split {dl_split} \
        --raw-dir data/raw \
        --receipt-dir data/p3 \
        --download-only

# 3. Sao lưu vào Google Drive Cache để không phải tải lại các lần sau
if DRIVE_CACHE.exists():
    for fname in ["testing.zip", "training.zip"]:
        src = RAW_DIR / fname
        dst = DRIVE_CACHE / fname
        if src.exists() and not dst.exists():
            print(f"Sao lưu {fname} sang {dst}...")
            shutil.copy2(src, dst)

print("\nTrạng thái file trong data/raw/:")
print(f"  - testing.zip:  {'✅ ĐÃ CÓ (' + str(TEST_ZIP.stat().st_size // (1024*1024)) + ' MB)' if TEST_ZIP.exists() else '❌ THIẾU'}")
print(f"  - training.zip: {'✅ ĐÃ CÓ (' + str(TRAIN_ZIP.stat().st_size // (1024*1024)) + ' MB)' if TRAIN_ZIP.exists() else '❌ THIẾU'}")


In [ ]:
# @title 2.2 Chuẩn hóa Source Receipts data/p3/*-source.json { display-mode: "form" }
import hashlib
import json
from pathlib import Path

P3_DIR = Path("/content/crossstage-stitch/data/p3")
RAW_DIR = Path("/content/crossstage-stitch/data/raw")
P3_DIR.mkdir(parents=True, exist_ok=True)

for split in ["testing", "training"]:
    receipt_file = P3_DIR / f"{split}-source.json"
    zip_file = RAW_DIR / f"{split}.zip"
    if zip_file.exists():
        print(f"Kiểm tra/Tính SHA-256 cho {zip_file.name}...")
        hasher = hashlib.sha256()
        with open(zip_file, "rb") as f:
            while chunk := f.read(2 * 1024 * 1024):
                hasher.update(chunk)
        digest = hasher.hexdigest().lower()
        
        # Chuẩn hóa receipt chính xác theo schema udis-source-v1
        clean_meta = {
            "schema_version": "udis-source-v1",
            "split": split,
            "author_source_url": "https://github.com/nie-lang/UnsupervisedDeepImageStitching",
            "download_url": f"https://drive.google.com/uc?id={'1unnBdPg9Vi12-1xXoUdzMub_XUq7yZz0' if split == 'testing' else '1CTKa42BRdXO2vdYSadaQkvgQ8DR6l49a'}",
            "sha256": digest,
            "checksum_origin": "user-pinned",
            "checksum_reference": f"Official UDIS-D Google Drive archive ({split}.zip)"
        }
        receipt_file.write_text(json.dumps(clean_meta, indent=2), encoding="utf-8")
        print(f"✅ {receipt_file.name}: SHA-256 = {digest[:16]}... (Split: {split})")
    else:
        print(f"⚠️ Chưa có file {zip_file.name}")


## 3. Thực thi Strict Ingestion: Giải mã PPM Nguyên bản & Kiểm chứng 1.105 Cặp Test (CH-01)

Quy tắc nghiêm ngặt của `udis_ingest`:
- Mode mặc định: `real` (từ chối synthetic fallback).
- Decode uint8 RGB nguyên bản, không nén, không sửa đổi geometry, lưu file `.ppm` và tính SHA-256 từng ảnh.
- Test split nạp đúng **1.105 cặp ảnh canonical** (`canonical_test: true`; bỏ qua tệp mồ côi `000001` do tác giả thiếu `input1/000001.jpg`). Bất kỳ sự thiếu hụt nào khác sẽ dừng với lỗi `count_error`.

In [ ]:
# @title 3.1 Chạy Ingestion cho Testing Split và Training Split { display-mode: "form" }
import shutil
from pathlib import Path

# Dọn dẹp thư mục dở dang nếu phiên chạy trước bị ngắt giữa chừng
for split in ["testing", "training"]:
    out_dir = Path(f"data/p3/{split}")
    if out_dir.exists() and not (out_dir / "data_manifest.json").exists():
        print(f"Dọn dẹp thư mục dở dang: {out_dir}")
        shutil.rmtree(out_dir)

print("=== Bắt đầu Ingestion Testing Split (1.105 cặp canonical) ===")
!python -m tools.data_prep.udis_ingest \
  --archive data/raw/testing.zip \
  --source-receipt data/p3/testing-source.json \
  --split testing \
  --output data/p3/testing \
  --verbose

print("\n=== Bắt đầu Ingestion Training Split ===")
!python -m tools.data_prep.udis_ingest \
  --archive data/raw/training.zip \
  --source-receipt data/p3/training-source.json \
  --split training \
  --output data/p3/training \
  --verbose


In [ ]:
# @title 3.2 Xác minh Ingestion Bất biến (--verify-only) { display-mode: "form" }
!python -m tools.data_prep.udis_ingest --output data/p3/testing --verify-only
!python -m tools.data_prep.udis_ingest --output data/p3/training --verify-only


## 4. Đóng băng Protocol: Split Đồ thị Liên thông, Grid 48 Cấu hình & Quality Policy (CH-01)

Module `freeze_protocol` thực hiện:
- Phân rã đồ thị connected components trên SHA-256 ảnh để tách rời hoàn toàn `train-fit` (250 cặp) và `train-val` (50+ cặp), ngăn chặn rò rỉ hình ảnh (data leakage).
- Đóng băng toàn bộ 1.105 cặp test trong `pairs_test.json`.
- Sinh `config_grid.json` gồm 48 cấu hình cartesian ($S \in \{0.25, 0.5, 0.75, 1.0\}$, $K \in \{500, 1000, 2000\}$, $R \in \{128, 256, 512, 1024\}$, $B=\text{feather}$).
- Thiết lập `quality_policy.json` chuẩn ($	ext{Coverage} \ge 0.95$, $\Delta\text{SSIM} \ge -0.02$, $\Delta\text{PSNR} \ge -0.5\text{ dB}$).
- Xuất `protocol_manifest.json` và khóa protocol.

In [ ]:
# @title 4.1 Thực thi Freeze Protocol { display-mode: "form" }
import shutil
from pathlib import Path

proto_dir = Path("data/p3/protocol")
if proto_dir.exists() and not (proto_dir / "protocol_manifest.json").exists():
    print(f"Dọn dẹp thư mục protocol dở dang: {proto_dir}")
    shutil.rmtree(proto_dir)

!python -m tools.data_prep.freeze_protocol \
  --training data/p3/training/data_manifest.json \
  --testing data/p3/testing/data_manifest.json \
  --seed 20261030 \
  --development-count 250 \
  --validation-count 50 \
  --output data/p3/protocol

# Xác minh tính toàn vẹn của protocol đã đóng băng
!python -m tools.data_prep.freeze_protocol --output data/p3/protocol --verify-only


## 5. Biên dịch CUDA Pipeline và Chạy 15/15 Kiểm thử CTest trên Tesla T4

Biên dịch toàn bộ hệ thống bằng CMake với cờ tối ưu hóa kiến trúc Tesla T4 `-DCMAKE_CUDA_ARCHITECTURES=75`.

In [ ]:
# @title 5.1 Biên dịch CMake Release (sm_75) và Kiểm thử CTest { display-mode: "form" }
!cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DCMAKE_CUDA_ARCHITECTURES=75
!cmake --build build -j$(nproc)
!ctest --test-dir build --output-on-failure


## 6. Thực thi Development GPU Sweep trên Tesla T4

Bạn có hai lựa chọn thuận tiện:
- **Lựa chọn 6.1A (Nhanh nhất - Dùng telemetry Tuấn đã chạy):** File nén mã nguồn đã chứa sẵn kết quả 15.000 dòng telemetry thực tế trên Colab T4 (`p3_dev25_results/results/`). Cell dưới đây sẽ nạp và phân tích ngay lập tức mà bạn không cần đợi GPU chạy lại.
- **Lựa chọn 6.1B (Chạy trực tiếp GPU Sweep):** Thực thi lệnh `tools.sweep.run_sweep` trên tập fit (250 cặp) và val (50 cặp) bằng GPU Tesla T4 để thu thập file CSV mới tinh.

In [ ]:
# @title 6.1 Lựa chọn A: Nạp và Khảo sát Telemetry Tuấn đã đo sẵn (p3_dev25_results) { display-mode: "form" }
import json
from pathlib import Path
import pandas as pd

RESULTS_DIR = Path("p3_dev25_results/results")
if RESULTS_DIR.exists():
    meta = json.loads((RESULTS_DIR / "meta.json").read_text(encoding="utf-8"))
    print("=== THÔNG TIN PHIÊN ĐO TELEMETRY TRÊN TESLA T4 CỦA TUẤN ===")
    print(f"  - GPU: {meta.get('gpu_name')} | VRAM: {meta.get('vram_total')}")
    print(f"  - Xung nhịp đồ họa: {meta.get('clock_graphics')} | Bộ nhớ: {meta.get('clock_mem')}")
    print(f"  - Tập dữ liệu: {meta.get('subset')} ({meta.get('split')}) | Số dòng: {meta.get('telemetry_rows')}")
    
    df_telemetry = pd.read_csv(RESULTS_DIR / "telemetry_results.csv")
    print(f"\nĐã nạp {len(df_telemetry)} dòng kết quả từ telemetry_results.csv")
    print("Phân bổ số lượt chạy theo chế độ:")
    print(df_telemetry['run_id'].value_counts())
    
    # Xem mẫu Fixed-Best config đã chọn
    fb_cfg = json.loads((RESULTS_DIR / "fixed_best_config.json").read_text(encoding="utf-8"))
    print(f"\nCấu hình Fixed-Best tạm tính: {fb_cfg[0]['config_id']}")
else:
    print("Chưa thấy p3_dev25_results/results/")


In [ ]:
# @title 6.2 Lựa chọn B: Chạy Live GPU Sweep trên train-fit và train-val (Nếu muốn đo trực tiếp) { display-mode: "form" }
from pathlib import Path
import shutil

FIT_DIR = Path("data/p3/sweep-fit")
VAL_DIR = Path("data/p3/sweep-val")

# 1. Quét trên train-fit (250 cặp ảnh)
print("=== Bắt đầu quét GPU trên train-fit ===")
!python -m tools.sweep.run_sweep \
  --manifest data/p3/protocol/pairs_train-fit.json \
  --policy data/p3/protocol/quality_policy.json \
  --ref-config S1.00_K2000_R1024_Bfeather \
  --out-dir data/p3/sweep-fit

if (FIT_DIR / "runs.csv").exists():
    shutil.copy2(FIT_DIR / "runs.csv", "data/p3/train-fit-runs.csv")
if (FIT_DIR / "qor.csv").exists():
    shutil.copy2(FIT_DIR / "qor.csv", "data/p3/train-fit-qor.csv")

# 2. Quét trên train-val (50+ cặp ảnh)
print("\n=== Bắt đầu quét GPU trên train-val ===")
!python -m tools.sweep.run_sweep \
  --manifest data/p3/protocol/pairs_train-val.json \
  --policy data/p3/protocol/quality_policy.json \
  --ref-config S1.00_K2000_R1024_Bfeather \
  --out-dir data/p3/sweep-val

if (VAL_DIR / "runs.csv").exists():
    shutil.copy2(VAL_DIR / "runs.csv", "data/p3/train-val-runs.csv")
if (VAL_DIR / "qor.csv").exists():
    shutil.copy2(VAL_DIR / "qor.csv", "data/p3/train-val-qor.csv")

print("\n✅ Đã hoàn thành Live Sweep trong data/p3/!")


## 7. Audit Reference & Chốt Cấu hình Fixed-Best (Hoàn tất đóng mốc CH-01)

Module `audit_development` đọc 4 file CSV từ development:
- Thẩm định độ tin cậy của cấu hình tham chiếu `S1.00_K2000_R1024_Bfeather`.
- Xếp hạng và chọn cấu hình `Fixed-Best` dựa trên tỷ lệ đạt QoR cao nhất và thời gian latency trung vị thấp nhất trên tập `train-fit`.
- Kiểm chứng chéo tính ổn định của cấu hình này trên tập `train-val`.
- Tuyệt đối từ chối và chặn dữ liệu test set (chống rò rỉ thông tin trước benchmark chính thức).

In [ ]:
# @title 7.1 Thực thi Audit Development và Xác minh { display-mode: "form" }
!python -m tools.data_prep.audit_development \
  --protocol data/p3/protocol \
  --fit-runs data/p3/train-fit-runs.csv \
  --fit-qor data/p3/train-fit-qor.csv \
  --val-runs data/p3/train-val-runs.csv \
  --val-qor data/p3/train-val-qor.csv \
  --output data/p3/development-audit.json \
  --verbose

# Xác minh lại tính bất biến của kết quả audit
!python -m tools.data_prep.audit_development \
  --protocol data/p3/protocol \
  --output data/p3/development-audit.json \
  --verify-only


In [ ]:
# @title 7.2 Xem Tóm tắt Kết quả Fixed-Best Đã Chọn { display-mode: "form" }
import json
from pathlib import Path

audit_file = Path("data/p3/development-audit.json")
if audit_file.exists():
    data = json.loads(audit_file.read_text(encoding="utf-8"))
    print("=== KẾT QUẢ AUDIT DEVELOPMENT ===")
    print(f"Trạng thái Reference: {data.get('reference_audit', {}).get('status')}")
    print(f"Cấu hình Fixed-Best chọn được: {data.get('selected_fixed_best')}")
    print(f"Tỷ lệ đạt QoR trên train-fit: {data.get('fit_metrics', {}).get('pass_rate')}")
    print(f"Tỷ lệ đạt QoR trên train-val: {data.get('val_metrics', {}).get('pass_rate')}")
    print("\n🎉 CH-01 ĐÃ HOÀN TẤT VÀ SẴN SÀNG CHO BENCHMARK CHÍNH THỨC & ABLATION A4/A5!")
else:
    print("Chưa tìm thấy data/p3/development-audit.json")


## 8. Đóng gói và Sao lưu Bundle `data/p3/`

Nén toàn bộ thư mục `data/p3/` (gồm protocol, manifest, các file CSV sweep và development audit report) thành một file ZIP để sao lưu an toàn sang Google Drive.

In [ ]:
# @title 8.1 Nén Bundle Protocol & Kết quả Development { display-mode: "form" }
import shutil
from pathlib import Path

P3_BUNDLE = Path("/content/p3_protocol_bundle")
shutil.make_archive(str(P3_BUNDLE), "zip", root_dir="/content/crossstage-stitch/data/p3")
zip_path = Path("/content/p3_protocol_bundle.zip")
print(f"✅ Đã tạo bundle thành công tại: {zip_path} ({zip_path.stat().st_size // (1024*1024)} MB)")

# Tùy chọn lưu vào Drive
if Path("/content/drive/MyDrive").exists():
    drive_dest = Path("/content/drive/MyDrive/p3_protocol_bundle.zip")
    shutil.copy2(zip_path, drive_dest)
    print(f"✅ Đã sao lưu sang Google Drive: {drive_dest}")


## 9. Thực thi Tiếp các Task Phase 3 của Chương (Ablation A4 & CH-05)

Khi đã có môi trường Tesla T4 và dữ liệu UDIS-D chuẩn bị xong, bạn có thể thực thi ngay hai task quan trọng do Chương chủ trì:
1. **Ablation A4 (Same-H Blending):** Đo đạc tách bạch GPU render interval vs Host invocation, kiểm chứng tính bất biến của alignment SSIM/PSNR giữa Feather, Multiband 3 và Multiband 5, đồng thời trích xuất các crop seam định tính.
2. **CH-05 (Synthetic Scaling 720p/1080p/4K):** Đo đạc độ co giãn thời gian và footprint bộ nhớ khi độ phân giải tăng vọt lên 4K.

In [ ]:
# @title 9.1 Chạy Ablation A4 (Same-H) và CH-05 (Synthetic Scaling) { display-mode: "form" }

print("=== 1. CHẠY THỰC NGHIỆM CH-05: SYNTHETIC SCALING 720p / 1080p / 4K ===")
!python -m tools.bench.stress_multires \
    --resolutions 720p 1080p 4K \
    --repeats 10 \
    --out-dir build/p3-colab/ch05_scaling

print("\n=== 2. CHẠY THỰC NGHIỆM ABLATION A4: SAME-HOMOGRAPHY BLENDING ===")
# Chạy thử nghiệm trên các cặp ảnh UDIS-D test set
if Path("data/p3/testing/data_manifest.json").exists():
    !python -m tools.bench.same_h \
        --manifest data/p3/testing/data_manifest.json \
        --modes feather multiband_3 multiband_5 \
        --repeats 10 \
        --out-dir build/p3-colab/a4_same_h
else:
    print("ℹ️ Cần hoàn tất Bước 3 (Ingestion) để có data/p3/testing/data_manifest.json trước khi chạy A4.")
